# EDA — (Exploratory Data Analysis) sur le dataset C-MAPSS (FD001)

**Objectif de ce notebook** : comprendre la structure du dataset C-MAPSS (FD001), identifier les capteurs
informatifs vs constants, visualiser les trajectoires de dégradation, et poser les bases (RUL, corrélations)
pour le feature engineering.

**Dataset** : FD001 — 1 condition opérationnelle, 1 mode de panne (dégradation du HPC), 100 moteurs en train,
100 moteurs en test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", 50)


: 

## 1. Chargement des données

Les fichiers C-MAPSS n'ont pas d'en-tête. Les colonnes sont, dans l'ordre :
`unit_number`, `time_in_cycles`, 3 réglages opérationnels (`op_setting_1..3`), et 21 mesures capteurs (`sensor_1..21`).


In [ ]:
index_names = ["unit_number", "time_in_cycles"]
setting_names = [f"op_setting_{i}" for i in range(1, 4)]
sensor_names = [f"sensor_{i}" for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

train = pd.read_csv("data/train_FD001.txt", sep=r"\s+", header=None, names=col_names)
test = pd.read_csv("data/test_FD001.txt", sep=r"\s+", header=None, names=col_names)
rul_test = pd.read_csv("data/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

print("Train:", train.shape)
print("Test :", test.shape)
print("RUL (test, ground truth):", rul_test.shape)
train.head()


## 2. Vue d'ensemble & valeurs manquantes

Premier réflexe sur n'importe quel dataset : types, valeurs manquantes, nombre d'unités.


In [ ]:
print(train.info())
print("\nValeurs manquantes par colonne :")
print(train.isna().sum().sum(), "valeurs manquantes au total (train)")
print("\nNombre de moteurs (train) :", train['unit_number'].nunique())
print("Nombre de moteurs (test)  :", test['unit_number'].nunique())


**Constat attendu** : pas de valeurs manquantes (dataset simulé, propre). C'est rare en industriel réel —
à mentionner dans le rapport comme une limite du dataset (pas de gestion de capteurs défaillants/manquants ici).


## 3. Distribution de la durée de vie des moteurs (train)

Chaque moteur tourne jusqu'à la panne. La durée de vie (nombre de cycles) varie d'un moteur à l'autre :
c'est notre première info sur la variabilité du problème.


In [ ]:
life = train.groupby("unit_number")["time_in_cycles"].max()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
life.plot(kind="bar", ax=ax[0], width=0.8)
ax[0].set_title("Durée de vie par moteur (train)")
ax[0].set_xlabel("unit_number")
ax[0].set_ylabel("cycles avant panne")
ax[0].set_xticks([])

sns.histplot(life, bins=20, ax=ax[1])
ax[1].set_title("Distribution des durées de vie")
ax[1].set_xlabel("cycles avant panne")
plt.tight_layout()
plt.show()

print(life.describe())


**À noter dans le journal de bord** : min, max, médiane de la durée de vie. Cette variance est justement
ce qui rend le problème difficile — un modèle doit généraliser à des dégradations plus ou moins rapides.


## 4. Calcul de la RUL (Remaining Useful Life) pour le train

Pour le train, on connaît le cycle de panne exact (dernier cycle enregistré). La RUL à un instant t est donc :

`RUL(t) = cycle_max_du_moteur - t`


In [ ]:
max_cycle = train.groupby("unit_number")["time_in_cycles"].transform("max")
train["RUL"] = max_cycle - train["time_in_cycles"]

train[["unit_number", "time_in_cycles", "RUL"]].head(10)


⚠️ **Piège classique (piecewise RUL)** : en début de vie, la RUL "vraie" décroît linéairement mais le
moteur ne montre quasiment aucun signe de dégradation — beaucoup de papiers de référence (dont le challenge
PHM 2008) plafonnent la RUL à une valeur max (ex: 125 cycles) plutôt que de la laisser croître indéfiniment
en arrière. À garder en tête pour la semaine 2 (feature engineering / target).


## 5. Réglages opérationnels

FD001 est censé n'avoir qu'**une seule** condition opérationnelle. Vérifions.


In [ ]:
print(train[setting_names].describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(setting_names):
    sns.histplot(train[col], bins=30, ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()


**Constat attendu** : très peu de variance sur les 3 réglages → confirme que FD001 = 1 seule condition
opérationnelle (contrairement à FD002/FD004 qui en ont 6). Sur FD001, on peut donc probablement ignorer ces
colonnes en feature engineering, ou les garder à titre de vérification.


## 6. Capteurs constants ou quasi-constants

Un capteur qui ne varie pas n'apporte aucune information pour prédire la dégradation. On les identifie via
l'écart-type (normalisé) et le nombre de valeurs uniques.


In [ ]:
sensor_std = train[sensor_names].std().sort_values()
sensor_nunique = train[sensor_names].nunique().sort_values()

summary = pd.DataFrame({"std": sensor_std, "n_unique": sensor_nunique.reindex(sensor_std.index)})
summary


In [ ]:
# seuil arbitraire mais standard : std quasi nulle -> capteur à exclure
constant_sensors = summary[summary["std"] < 1e-4].index.tolist()
print("Capteurs constants/quasi-constants détectés :", constant_sensors)

useful_sensors = [s for s in sensor_names if s not in constant_sensors]
print(f"\n{len(useful_sensors)} capteurs informatifs sur {len(sensor_names)} au total.")


**À écrire dans le rapport** : liste exacte des capteurs exclus et pourquoi. C'est une conclusion
actionnable directe pour le feature engineering de la semaine 2 (on ne gardera que `useful_sensors`).


## 7. Trajectoires de dégradation — visualiser quelques moteurs

L'idée : tracer plusieurs capteurs informatifs en fonction du cycle, pour quelques moteurs, et voir si une
tendance de dégradation est visible à l'œil nu avant la panne.


In [ ]:
sample_units = [1, 2, 3, 4, 5]
sensors_to_plot = useful_sensors[:6]  # les 6 premiers capteurs informatifs

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, sensor in enumerate(sensors_to_plot):
    for unit in sample_units:
        unit_data = train[train["unit_number"] == unit]
        axes[i].plot(unit_data["time_in_cycles"], unit_data[sensor], alpha=0.7, label=f"unit {unit}")
    axes[i].set_title(sensor)
    axes[i].set_xlabel("cycle")

axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()


**Question à te poser pour chaque capteur** : la tendance est-elle monotone (dégradation claire) ou
bruitée sans tendance visible ? Note dans ton journal quels capteurs semblent les plus prometteurs
(typiquement sur C-MAPSS : sensor_2, sensor_3, sensor_4, sensor_7, sensor_11, sensor_15 montrent une
tendance assez nette — à vérifier toi-même sur les graphes ci-dessus plutôt que de prendre ça pour acquis).


## 8. Capteurs vs RUL — la relation qu'on veut modéliser

Plutôt que capteur vs cycle, on regarde capteur vs RUL (0 = panne). C'est la relation que le modèle devra
apprendre.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

# échantillon de lignes pour ne pas surcharger le scatter
sample = train.sample(5000, random_state=42)

for i, sensor in enumerate(sensors_to_plot):
    axes[i].scatter(sample["RUL"], sample[sensor], s=3, alpha=0.3)
    axes[i].set_title(sensor)
    axes[i].set_xlabel("RUL")
    axes[i].invert_xaxis()  # RUL décroissante vers la panne, plus intuitif à lire de gauche à droite

plt.tight_layout()
plt.show()


## 9. Corrélations entre capteurs

Beaucoup de capteurs sont probablement redondants (même phénomène physique mesuré différemment) — utile pour
justifier une éventuelle PCA en semaine 2.


In [ ]:
corr = train[useful_sensors].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Corrélation entre capteurs informatifs")
plt.tight_layout()
plt.show()


In [ ]:
# paires de capteurs très corrélées (redondance potentielle)
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]  # enlever diagonale
corr_pairs = corr_pairs[~corr_pairs.index.duplicated()]
print("Top 10 paires de capteurs les plus corrélées :")
print(corr_pairs.head(10))


**À noter** : si plusieurs paires dépassent 0.9-0.95 de corrélation, c'est un argument concret pour la
PCA ou pour n'en garder qu'un par groupe redondant — à documenter dans le rapport plutôt que d'appliquer une
PCA "par défaut" sans justification.


## 10. Corrélation capteurs ↔ RUL

Quels capteurs sont, en corrélation linéaire simple, les plus informatifs pour prédire la RUL ? (Attention :
une faible corrélation linéaire ne veut pas dire "pas informatif" — la relation peut être non-linéaire, un
arbre ou un LSTM peut quand même l'exploiter.)


In [ ]:
rul_corr = train[useful_sensors + ["RUL"]].corr()["RUL"].drop("RUL").sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 6))
rul_corr.plot(kind="barh")
plt.title("Corrélation linéaire (capteur, RUL)")
plt.xlabel("corrélation de Pearson")
plt.tight_layout()
plt.show()

rul_corr


## 11. Synthèse — conclusions actionnables pour la semaine 2

À compléter toi-même une fois les graphes ci-dessus inspectés (ne recopie pas une conclusion générique —
vérifie sur *tes* résultats) :

- **Capteurs à exclure** (constants) : voir `constant_sensors` calculé en section 6.
- **Capteurs les plus prometteurs** (tendance visible + corrélation avec RUL) : à lister à partir des sections 7, 8, 10.
- **Redondance entre capteurs** : paires très corrélées identifiées en section 9 → candidates à une réduction
  (PCA, ou sélection manuelle d'un capteur par groupe).
- **FD001 = condition opérationnelle unique** : confirmé en section 5, donc pas besoin de normaliser par
  condition opérationnelle sur ce sous-set (contrairement à FD002/FD004).
- **Pas de valeurs manquantes** : dataset simulé et propre — à mentionner comme limite par rapport à un cas
  industriel réel dans le rapport final.
- **Distribution de la durée de vie** : min/max/médiane relevés en section 3 — utile pour choisir un seuil de
  RUL max (piecewise linear RUL) en semaine 2.

**Prochaine étape (semaine 2)** : fenêtres glissantes (rolling mean/std) sur les capteurs retenus,
définition du seuil de plafonnement de la RUL, et split temporel train/val par unité (pas par ligne !).
